In [1]:
import os
# Cài đặt pyvi để phân từ tiếng Việt (Bắt buộc cho PhoBERT để đạt điểm cao)
os.system('pip install pyvi')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from pyvi import ViTokenizer
import warnings
warnings.filterwarnings('ignore')
from tqdm import tqdm
import random

# ==========================================
# 1. SET SEED & CONFIGURATION
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

class Config:
    model_name = "vinai/phobert-base"
    max_len = 120    # GIẢM XUỐNG 120 ĐỂ TĂNG TỐC ĐỘ (Feedback sinh viên thường ngắn)
    batch_size = 32  # TĂNG LÊN 32 VÌ MAX_LEN ĐÃ GIẢM
    epochs = 4       # Giảm xuống 4 để tránh Overfitting
    learning_rate = 2e-5
    weight_decay = 0.01
    n_folds = 5
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    num_classes = 3  # Sentiment: 0, 1, 2
    num_topics = 4   # Topic: 0, 1, 2, 3
    
    warmup_ratio = 0.1
    topic_loss_weight = 0.3  # Giảm bớt trọng số topic để model tập trung vào sentiment
    
config = Config()
print(f"Using device: {config.device}")

# ==========================================
# 2. LOAD & PREPROCESS DATA
# ==========================================
try:
    train_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/train.csv')
    test_df = pd.read_csv('/kaggle/input/competitions/midtermNLP01/test.csv')
except FileNotFoundError:
    train_df = pd.read_csv('train.csv')
    test_df = pd.read_csv('test.csv')

# Tiền xử lý: Phân từ tiếng Việt (RẤT QUAN TRỌNG CHO PHO-BERT)
print("Đang phân từ tiếng Việt (Word Segmentation)...")
train_df['sentence'] = train_df['sentence'].apply(lambda x: ViTokenizer.tokenize(str(x)))
test_df['sentence'] = test_df['sentence'].apply(lambda x: ViTokenizer.tokenize(str(x)))

# Tính toán Class Weights để xử lý mất cân bằng dữ liệu
sent_weights = compute_class_weight('balanced', classes=np.unique(train_df['sentiment']), y=train_df['sentiment'])
sent_weights = torch.tensor(sent_weights, dtype=torch.float).to(config.device)

topic_weights = compute_class_weight('balanced', classes=np.unique(train_df['topic']), y=train_df['topic'])
topic_weights = torch.tensor(topic_weights, dtype=torch.float).to(config.device)

# ==========================================
# 3. DATASET
# ==========================================
class JointSentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        sentence = str(self.df.iloc[idx]['sentence'])
        
        encoding = self.tokenizer(
            sentence,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        if self.is_train:
            item['sentiment'] = torch.tensor(self.df.iloc[idx]['sentiment'], dtype=torch.long)
            item['topic'] = torch.tensor(self.df.iloc[idx]['topic'], dtype=torch.long)
            
        return item

# ==========================================
# 4. MODEL CẢI TIẾN
# ==========================================
class JointPhoBERT(nn.Module):
    def __init__(self, model_name, num_topics, num_sentiments):
        super(JointPhoBERT, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        
        self.topic_classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(self.bert.config.hidden_size, num_topics)
        )
        
        # Thêm LayerNorm để ổn định việc ghép nối features
        self.layer_norm = nn.LayerNorm(self.bert.config.hidden_size + num_topics)
        
        self.sentiment_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.bert.config.hidden_size + num_topics, 128),
            nn.GELU(),
            nn.Linear(128, num_sentiments)
        )
        
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :] # CLS token
        
        # Dự đoán Topic
        topic_logits = self.topic_classifier(pooled_output)
        topic_probs = torch.softmax(topic_logits, dim=1)
        
        # Nối và chuẩn hóa
        combined_features = torch.cat((pooled_output, topic_probs), dim=1)
        combined_features = self.layer_norm(combined_features)
        
        # Dự đoán Sentiment
        sentiment_logits = self.sentiment_classifier(combined_features)
        
        return topic_logits, sentiment_logits

# ==========================================
# 5. TRAINING LOOP TỐI ƯU VỚI AMP (Tăng tốc GPU)
# ==========================================
def train_epoch(model, data_loader, optimizer, scheduler, crit_topic, crit_sent, device, scaler):
    model.train()
    losses = []
    all_preds = []
    all_labels = []
    
    progress_bar = tqdm(data_loader, desc='Training', leave=False)
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_sent = batch['sentiment'].to(device)
        labels_topic = batch['topic'].to(device)
        
        optimizer.zero_grad()
        
        # Sử dụng Automatic Mixed Precision (AMP) giúp tăng tốc x2 lần trên GPU
        with torch.cuda.amp.autocast():
            topic_logits, sentiment_logits = model(input_ids, attention_mask)
            loss_topic = crit_topic(topic_logits, labels_topic)
            loss_sent = crit_sent(sentiment_logits, labels_sent)
            loss = loss_sent + config.topic_loss_weight * loss_topic
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        losses.append(loss.item())
        _, preds_sent = torch.max(sentiment_logits, dim=1)
        all_preds.extend(preds_sent.cpu().numpy())
        all_labels.extend(labels_sent.cpu().numpy())
        progress_bar.set_postfix({'loss': np.mean(losses)})
    
    return np.mean(losses), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='weighted')

def eval_epoch(model, data_loader, crit_topic, crit_sent, device):
    model.eval()
    losses = []
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        progress_bar = tqdm(data_loader, desc='Evaluating', leave=False)
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_sent = batch['sentiment'].to(device)
            labels_topic = batch['topic'].to(device)
            
            with torch.cuda.amp.autocast():
                topic_logits, sentiment_logits = model(input_ids, attention_mask)
                loss_topic = crit_topic(topic_logits, labels_topic)
                loss_sent = crit_sent(sentiment_logits, labels_sent)
                loss = loss_sent + config.topic_loss_weight * loss_topic
            
            losses.append(loss.item())
            _, preds_sent = torch.max(sentiment_logits, dim=1)
            all_preds.extend(preds_sent.cpu().numpy())
            all_labels.extend(labels_sent.cpu().numpy())
            
    return np.mean(losses), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='weighted')

def predict(model, data_loader, device):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Predicting'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            with torch.cuda.amp.autocast():
                _, sentiment_logits = model(input_ids, attention_mask)
            _, preds = torch.max(sentiment_logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
    return all_preds

# ==========================================
# 6. MAIN EXECUTION (K-FOLD)
# ==========================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)

test_dataset = JointSentimentDataset(test_df, tokenizer, config.max_len, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size*2, shuffle=False) # Predict có thể dùng batch lớn hơn

skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)
X = train_df['sentence'].values
y = train_df['sentiment'].values

fold_predictions = []
val_scores = []

# Tích hợp Loss Function có đánh trọng số Class Weights + Label Smoothing
crit_topic = nn.CrossEntropyLoss(weight=topic_weights, label_smoothing=0.05)
crit_sent = nn.CrossEntropyLoss(weight=sent_weights, label_smoothing=0.05)

# Tích hợp scaler cho AMP (Tăng tốc)
scaler = torch.cuda.amp.GradScaler()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n{'='*50}\nFold {fold + 1}/{config.n_folds}\n{'='*50}")
    
    train_fold = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold = train_df.iloc[val_idx].reset_index(drop=True)
    
    train_dataset = JointSentimentDataset(train_fold, tokenizer, config.max_len, is_train=True)
    val_dataset = JointSentimentDataset(val_fold, tokenizer, config.max_len, is_train=True)
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    
    model = JointPhoBERT(config.model_name, config.num_topics, config.num_classes).to(config.device)
    
    # Optimizer (loại bỏ weight_decay cho LayerNorm và Bias để tránh underfitting)
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': config.weight_decay},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=config.learning_rate)
    
    total_steps = len(train_loader) * config.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * config.warmup_ratio), num_training_steps=total_steps)
    
    best_val_f1 = 0
    best_model_state = None
    
    for epoch in range(config.epochs):
        print(f"Epoch {epoch + 1}/{config.epochs}")
        train_loss, train_acc, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, crit_topic, crit_sent, config.device, scaler)
        print(f"Train - Loss: {train_loss:.4f}, F1: {train_f1:.4f}")
        
        val_loss, val_acc, val_f1 = eval_epoch(model, val_loader, crit_topic, crit_sent, config.device)
        print(f"Val   - Loss: {val_loss:.4f}, F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"--> Mới tạo kỷ lục F1: {val_f1:.4f}")
            
    model.load_state_dict(best_model_state)
    model = model.to(config.device)
    val_scores.append(best_val_f1)
    
    fold_pred = predict(model, test_loader, config.device)
    fold_predictions.append(fold_pred)

print(f"\nCross-validation scores: {val_scores}")
print(f"Mean CV F1: {np.mean(val_scores):.4f} (+/- {np.std(val_scores):.4f})")

# ==========================================
# 7. ENSEMBLE SUBMISSION
# ==========================================
fold_predictions = np.array(fold_predictions)

# Dùng Weighted Ensemble thay vì Majority Voting thuần túy sẽ chính xác hơn
weights = np.array(val_scores) / np.sum(val_scores)
weighted_predictions = []
for i in range(fold_predictions.shape[1]):
    weighted_votes = np.zeros(config.num_classes)
    for fold_idx in range(len(fold_predictions)):
        weighted_votes[fold_predictions[fold_idx, i]] += weights[fold_idx]
    weighted_predictions.append(np.argmax(weighted_votes))

submission = pd.DataFrame({'id': test_df['id'], 'sentiment': weighted_predictions})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print("\nĐã lưu file: /kaggle/working/submission.csv")
print(submission['sentiment'].value_counts().sort_index())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 53.8 MB/s eta 0:00:00
Using device: cuda
Đang phân từ tiếng Việt (Word Segmentation)...
Loading tokenizer...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Val   - Loss: 0.9608, F1: 0.9353
--> Mới tạo kỷ lục F1: 0.9353
Epoch 4/4


Train - Loss: 0.8038, F1: 0.9589


Val   - Loss: 0.9667, F1: 0.9354
--> Mới tạo kỷ lục F1: 0.9354


Predicting: 100%|██████████| 76/76 [00:08<00:00,  9.31it/s]



Cross-validation scores: [0.9443093832967351, 0.9412840454610437, 0.9369898326395592, 0.9410369415334792, 0.9353711871778826]
Mean CV F1: 0.9398 (+/- 0.0032)

Đã lưu file: /kaggle/working/submission.csv
sentiment
0    2215
1     194
2    2444
Name: count, dtype: int64
